[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cto-school/ml-ctoschool/blob/main/ml-00-numpy-essentials/01-numpy-arrays-and-attributes.ipynb)

# NumPy Arrays — Creation, Attributes, and Data Types
### Module 00 — Notebook 1

Welcome! This is the **very first step** on your journey from Python programmer to machine learning practitioner.

By the end of this notebook you will be able to:
- Explain *why* NumPy is the foundation of nearly every ML/data-science library in Python
- Create arrays in a dozen different ways (and know *when* to use each)
- Read and interpret array **attributes** — shape, ndim, size, dtype — like a pro
- Build the kind of arrays you will encounter in real ML pipelines

> **How to use this notebook:** Read every markdown cell, run every code cell, and attempt every exercise *before* peeking at the solution. Learning happens when you struggle a little!

Let's dive in! 🚀

---
## Section 1: Why NumPy?

### What is NumPy?

**NumPy** (Numerical Python) is a library that gives Python the ability to work with large arrays and matrices of numbers *efficiently*. It was first released in 2005 and has since become the **bedrock** of the entire Python data-science ecosystem.

Here is the key thing to understand:

> **Every major ML library — scikit-learn, TensorFlow, PyTorch, pandas, matplotlib — is built on top of NumPy arrays.**

When you call `model.fit(X, y)` in scikit-learn, `X` and `y` are NumPy arrays (or get converted into them). When you load an image in OpenCV, you get a NumPy array. When you compute gradients in PyTorch, the underlying data lives in a structure inspired by NumPy.

So learning NumPy is not optional — it is **prerequisite knowledge** for everything that follows.

### The Key Insight: Vectorized Operations vs Python Loops

Plain Python is an *interpreted* language. Looping over millions of numbers in Python is **slow** because every iteration goes through the interpreter. NumPy solves this by:

1. Storing data in contiguous blocks of memory (like C arrays)
2. Executing operations in pre-compiled C/Fortran code under the hood
3. Operating on *entire arrays at once* (this is called **vectorization**)

Let's see the speed difference with our own eyes.

In [ ]:
import numpy as np  # standard convention — ALWAYS import numpy as np
import time           # we'll use this to measure execution time

# --- Experiment: add two collections of 1,000,000 numbers ---

size = 1_000_000  # one million elements

# --- Method 1: Pure Python lists ---
list_a = list(range(size))  # [0, 1, 2, ..., 999999]
list_b = list(range(size))  # same thing

start = time.time()                              # start the clock
list_result = [a + b for a, b in zip(list_a, list_b)]  # element-wise add via list comprehension
python_time = time.time() - start                # stop the clock

print(f"Python list comprehension : {python_time:.4f} seconds")

# --- Method 2: NumPy arrays ---
arr_a = np.array(list_a)   # convert list to NumPy array
arr_b = np.array(list_b)

start = time.time()                # start the clock
arr_result = arr_a + arr_b         # element-wise add — one line, no loop!
numpy_time = time.time() - start   # stop the clock

print(f"NumPy vectorized addition : {numpy_time:.4f} seconds")
print(f"\nNumPy is ~{python_time / numpy_time:.0f}x faster! 🔥")
print(f"\nFirst 5 results (Python) : {list_result[:5]}")
print(f"First 5 results (NumPy)  : {arr_result[:5]}")

### Why does this matter for ML?

In machine learning you routinely work with:
- Datasets with **millions of rows** and dozens (or hundreds) of columns
- Weight matrices inside neural networks that get updated **thousands of times** during training
- Image tensors where a single batch might be 32 images × 3 channels × 224 × 224 pixels = **~4.8 million numbers**

If each of those operations were a Python `for` loop, training a model would take *days* instead of *minutes*. NumPy (and its GPU cousins like CuPy and PyTorch) make modern ML computationally feasible.

---

---
## Section 2: Scalars, Vectors, Matrices, and Tensors — The Building Blocks of ML

Before we start creating arrays, we need to understand **what these arrays actually represent**. In ML, arrays are not just containers of numbers — they are the mathematical objects that every algorithm is built on.

There are exactly four types, forming a hierarchy from simple to complex:

### Scalar — A Single Number (0D)

A scalar is just one number. A temperature reading (72°F), a model's accuracy (0.94), a learning rate (0.001) — these are all scalars.

Every loss value your model computes during training is a scalar. Every threshold you set for a classifier is a scalar. They seem trivial, but they are the atoms that everything else is made of.

In [ ]:
# --- Scalar: a single number ---
learning_rate = np.float64(0.001)
print("Value:", learning_rate)
print("Shape:", learning_rate.shape)   # () — empty tuple, zero dimensions
print("ndim: ", learning_rate.ndim)     # 0
print("This is a 0D tensor (a scalar).")

### Vector — An Ordered List of Numbers, a Point in Space (1D)

A vector is an ordered sequence of numbers. But it is much more than a "list" — it represents **a point in space** or **a direction with magnitude**.

**The key insight for ML: every data sample you will ever work with IS a vector.**

Consider a house described by three features:
- Square footage: 1500
- Bedrooms: 3
- Price: $350,000

This house is the vector `[1500, 3, 350000]` — a single **point in 3-dimensional "house space"**. Every house in your dataset is another point in this same space. Your entire dataset is a cloud of points.

```
A single house: [1500, 3, 350000]
                   |    |     |
                   |    |     +-- axis 2 (price)
                   |    +-------- axis 1 (bedrooms)
                   +------------- axis 0 (sqft)

This is ONE point in a 3-dimensional feature space.
```

**What ML actually does with vectors:**
- **Classification** draws boundaries between clusters of points (vectors) in feature space
- **Regression** finds a surface that fits through the cloud of points
- **Clustering** groups nearby points together
- **Dimensionality reduction** projects points from high-dimensional space to lower dimensions while preserving structure

**Mathematical properties that matter for ML:**
- **Magnitude (length):** `|v| = sqrt(v1² + v2² + ... + vn²)` — how "big" the vector is. Used in regularization and normalization.
- **Direction:** Where the vector points in space. Two vectors can have the same magnitude but point in completely different directions.
- **Dot product:** `a · b = |a| |b| cos(θ)` — measures alignment between two vectors. This is the fundamental operation inside every neuron in a neural network.

In [ ]:
# --- Vector: a 1D array = a point in feature space ---

# A house described by 3 features: [sqft, bedrooms, price]
house = np.array([1500, 3, 350000])
print("House vector:", house)
print("Shape:", house.shape)   # (3,) — 3 features = a point in 3D space
print("ndim: ", house.ndim)     # 1
print()

# Magnitude (length) of the vector — how "far" this point is from the origin
magnitude = np.sqrt(np.sum(house ** 2))
print(f"Magnitude of house vector: {magnitude:.2f}")
print("(This is the Euclidean distance from the origin to this point in 3D space)")
print()

# An MNIST image flattened into a vector — a point in 784-dimensional space
mnist_image = np.random.rand(784)   # 28x28 = 784 pixel values
print(f"MNIST image as vector: shape {mnist_image.shape} — a point in 784D space!")
print("Every image is a point. Similar images are nearby points. That's how classification works.")

### Matrix — A Grid of Numbers (Two Interpretations) (2D)

A matrix is a 2D grid of numbers. What makes matrices powerful is that **they have two completely different interpretations in ML**, and both are essential.

**Interpretation 1 — A collection of vectors (your dataset):**

When you stack multiple sample vectors as rows, you get a matrix. Your entire dataset is a matrix.

```
Dataset X with shape (4, 3):    4 samples, 3 features each

         sqft  beds  price
House 0 [1500,   3, 350000]    <-- each row is one sample vector
House 1 [2200,   4, 520000]
House 2 [ 800,   1, 180000]
House 3 [1800,   3, 410000]
```

**Interpretation 2 — A linear transformation:**

A matrix can also act as a **function** that transforms vectors. When you multiply a matrix by a vector, you get a new vector — the matrix has rotated, scaled, stretched, or projected the original.

```
Transformation W with shape (3, 2):    maps 3D vectors to 2D vectors

Input vector (3 features)  -->  W  -->  Output vector (2 values)
     [1500, 3, 350000]     --> [3,2] -->  [y1, y2]
```

This is exactly what happens in a neural network layer: the weight matrix `W` transforms an input vector into an output vector. Training the network means finding the right numbers in `W`.

In [ ]:
# --- Matrix Interpretation 1: A dataset (collection of vectors) ---

# 4 houses, each described by 3 features
dataset = np.array([
    [1500,  3, 350000],    # House 0 — one point in 3D space
    [2200,  4, 520000],    # House 1 — another point
    [ 800,  1, 180000],    # House 2
    [1800,  3, 410000],    # House 3
])
print("Dataset (matrix of sample vectors):")
print(dataset)
print(f"Shape: {dataset.shape} \u2014 {dataset.shape[0]} samples, {dataset.shape[1]} features")
print(f"Each row is a vector (point) in {dataset.shape[1]}D feature space")
print()



In [ ]:
# Extract a single sample — it's a vector!
single_house = dataset[0]
print(f"Row 0 extracted: {single_house} \u2014 shape {single_house.shape} \u2014 back to a vector")
print()



In [ ]:
np.random.randn(5, 10) * 4.01

In [ ]:
np.random.randn(3, 2)

In [ ]:
# --- Matrix Interpretation 2: A transformation (weight matrix) ---

# A weight matrix that maps 3 input features to 2 output values
W = np.random.randn(3, 2) * 0.01   # small random weights
print("Weight matrix W (a transformation):")
print(W)
print(f"Shape: {W.shape} \u2014 maps {W.shape[0]}D input to {W.shape[1]}D output")
print()


In [137]:

# Apply the transformation: input vector @ weight matrix = output vector
output = single_house @ W
print(f"Input:  {single_house}  (shape {single_house.shape})")
print(f"Output: {output}  (shape {output.shape})")
print("The weight matrix transformed a 3D vector into a 2D vector!")

Input:  [  1500      3 350000]  (shape (3,))
Output: [-4200.09184635  3905.47528889]  (shape (2,))
The weight matrix transformed a 3D vector into a 2D vector!


### Tensor — The Generalization to Any Number of Dimensions (nD)

A tensor is the general term for any n-dimensional array of numbers. Scalars, vectors, and matrices are all special cases:

```
Scalar:  0D tensor    shape ()           a single number
Vector:  1D tensor    shape (n,)         a list of numbers
Matrix:  2D tensor    shape (m, n)       a grid of numbers
3D:      3D tensor    shape (a, b, c)    a "cube" of numbers
4D:      4D tensor    shape (a, b, c, d) a collection of cubes
```

The word "tensor" is why Google's ML framework is called **TensorFlow** — data flows through the computation graph as tensors.

**Where higher-dimensional tensors appear in ML:**
- **3D** `(batch, timesteps, features)` — a batch of time series sequences
- **3D** `(height, width, channels)` — a single color image (224×224×3 for RGB)
- **4D** `(batch, height, width, channels)` — a batch of color images fed to a CNN

In [138]:
# --- The full hierarchy: scalar -> vector -> matrix -> 3D tensor -> 4D tensor ---

scalar = np.float64(42.0)
vector = np.array([[[1],[3],[5]],[[1],[3],[5]]])
matrix = np.array([[1, 2],[3, 4],[7,8]])
tensor_3d = np.random.rand(3, 28, 28)          # 3 grayscale images
tensor_4d = np.random.rand(32, 224, 224, 3)    # batch of 32 color images

print("The Tensor Hierarchy:")
print(f"  Scalar  -> shape {str(scalar.shape):15s}  ndim {scalar.ndim}  -- e.g., learning rate")
print(f"  Vector  -> shape {str(vector.shape):15s}  ndim {vector.ndim}  -- e.g., one data sample")
print(f"  Matrix  -> shape {str(matrix.shape):15s}  ndim {matrix.ndim}  -- e.g., full dataset or weight matrix")
print(f"  3D      -> shape {str(tensor_3d.shape):15s}  ndim {tensor_3d.ndim}  -- e.g., batch of grayscale images")
print(f"  4D      -> shape {str(tensor_4d.shape):15s}  ndim {tensor_4d.ndim}  -- e.g., batch of color images")
print()
print(f"Memory for 4D tensor: {tensor_4d.nbytes / 1e6:.1f} MB -- just ONE batch of images!")

The Tensor Hierarchy:
  Scalar  -> shape ()               ndim 0  -- e.g., learning rate
  Vector  -> shape (2, 3, 1)        ndim 3  -- e.g., one data sample
  Matrix  -> shape (3, 2)           ndim 2  -- e.g., full dataset or weight matrix
  3D      -> shape (3, 28, 28)      ndim 3  -- e.g., batch of grayscale images
  4D      -> shape (32, 224, 224, 3)  ndim 4  -- e.g., batch of color images

Memory for 4D tensor: 38.5 MB -- just ONE batch of images!


### The Mental Shift: From "Table of Numbers" to "Cloud of Points"

This is the single most important mental model for understanding ML:

> When you see `shape (1000, 784)`, do **not** think "a 2D array with 1000 rows and 784 columns."
> Think: **"1000 data points, each living in 784-dimensional space."**

That shift — from "table of numbers" to "cloud of points in high-dimensional space" — is what separates someone who *uses* NumPy from someone who *understands* ML.

Every concept you learn from here on — distances, dot products, projections, transformations, gradients — operates on this mental model of vectors as points in space.

---

## Section 3: Creating Arrays

There are many ways to create NumPy arrays. We will cover them all, and for each one we will explain **when and why** you would use it in practice.

### 3.1 — From Python Lists

The most basic way: hand `np.array()` a Python list.

In [139]:
# --- Example 1: A simple 1-D array (a vector) ---
grades = np.array([85, 92, 78, 95, 88,88])   # five exam scores
print("1-D array (grades):", grades)
print("Type:", type(grades))                # <class 'numpy.ndarray'>

1-D array (grades): [85 92 78 95 88 88]
Type: <class 'numpy.ndarray'>


In [140]:
# --- Example 2: A 1-D array of floats ---
temperatures = np.array([36.6, 37.1, 36.8, 38.2, 36.5])  # body temperatures in °C
print("1-D float array (temperatures):", temperatures)
print("Notice the dtype is float64 (NumPy auto-detects):", temperatures.dtype)

1-D float array (temperatures): [36.6 37.1 36.8 38.2 36.5]
Notice the dtype is float64 (NumPy auto-detects): float64


In [141]:
# --- Example 3: A 1-D array with mixed int/float — NumPy UPCASTS ---
mixed = np.array(1, 2, 3,'a',3.14159)  # one float forces everything to float
print("Mixed input becomes all floats:", mixed)
print("dtype:", mixed.dtype)  # float64 — NumPy chose the 'safest' type

TypeError: array() takes from 1 to 2 positional arguments but 5 were given

#### Creating 2-D Arrays (Matrices)

A 2-D array is a **matrix** — rows and columns. In ML, your dataset is almost always a 2-D array where each row is a *sample* and each column is a *feature*.

In [ ]:
# --- Example 1: A 3x3 matrix ---
matrix = np.array([
    [1, 2, 3],    # row 0
    [4, 5, 6],    # row 1
    [7, 8, 9]     # row 2
])
print("2-D array (matrix):")
print(matrix)
print("Shape:", matrix.shape)  # (3, 3) — 3 rows, 3 columns

In [ ]:
# --- Example 2: Student grades — 4 students, 3 subjects ---
# Each row = one student, each column = one subject (math, science, english)
student_grades = np.array([
    [85, 92, 78],   # student 0
    [90, 88, 95],   # student 1
    [72, 80, 85],   # student 2
    [95, 97, 93]    # student 3
])
print("Student grades matrix (4 students × 3 subjects):")
print(student_grades)
print("Shape:", student_grades.shape)  # (4, 3)

In [144]:
# --- Example 3: RGB pixel values for a tiny 2x2 image ---
# In image processing, each pixel has 3 values: Red, Green, Blue (0-255)
tiny_image = np.array([
    [[255, 0, 0,0],   [0, 255, 0,0]],    # row 0: red pixel, green pixel
    [[0, 0, 255,0],   [255, 255, 0,0]] ,
    [[0, 0, 255,0],   [255, 255, 0,0]] ,   # row 1: blue pixel, yellow pixel
])# row 1: blue pixel, yellow pixel

print("Tiny 2x2 RGB image:")
print(tiny_image)
print("Shape:", tiny_image.shape)  # (2, 2, 3) — height, width, channels

Tiny 2x2 RGB image:
[[[255   0   0   0]
  [  0 255   0   0]]

 [[  0   0 255   0]
  [255 255   0   0]]

 [[  0   0 255   0]
  [255 255   0   0]]]
Shape: (3, 2, 4)


### 3.2 — Special Constructor Functions

NumPy provides convenience functions that create arrays filled with specific values. Each has a clear use case in ML.

#### `np.zeros` — Create an array filled with 0s
**ML use cases:** initializing weight matrices, creating placeholder arrays for results, padding sequences.

In [152]:
# --- Example 1: 1-D zeros ---
z1 = np.zeros(5)              # 5 zeros
print("1-D zeros:", z1)       # [0. 0. 0. 0. 0.]  (float64 by default)

# --- Example 2: 2-D zeros (like an empty weight matrix) ---
z2 = np.zeros((3, 4))         # 3 rows × 4 columns of zeros
print("\n2-D zeros (3×4):")
print(z2)

# --- Example 3: Zeros with a specific dtype ---
z3 = np.zeros((2, 3), dtype=np.int32)   # integer zeros (saves memory)
print(\nInteger zeros:")
print(z3)
print("dtype:", z3.dtype)

print("\n💡 ML context: You might initialize a neural network's weight matrix")
print("   as zeros before training begins (though random init is usually better).")

1-D zeros: [0. 0. 0. 0. 0.]

2-D zeros (3×4):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
10
[[0 0 0]
 [0 0 0]]
dtype: int32

💡 ML context: You might initialize a neural network's weight matrix
   as zeros before training begins (though random init is usually better).


#### `np.ones` — Create an array filled with 1s
**ML use cases:** bias term initialization, creating masks, scaling arrays.

In [158]:
# --- Example 1: 1-D ones ---
o1 = np.ones(4)              # [1. 1. 1. 1.]
print("1-D ones:", o1)

# --- Example 2: Ones matrix (like a bias vector) ---
bias = np.ones((1, 5))       # 1 row × 5 features — a bias term
print("\nBias vector (1×5):", bias)

# --- Example 3: Integer ones for a binary mask ---
mask = np.ones((3, 3), dtype=np.int8)   # tiny mask (8-bit integers save memory)
print("\nBinary mask (3×3):")
print(mask)
print("dtype:", mask.dtype)

print("\n💡 ML context: Bias terms in neural networks are often initialized to ones")
print("   (or small constants) so every neuron starts with some activation.")

1-D ones: [1. 1. 1. 1.]

Bias vector (1×5): [[1. 1. 1. 1. 1.]]

Binary mask (3×3):
[[1 1 1]
 [1 1 1]
 [1 1 1]]
dtype: int8

💡 ML context: Bias terms in neural networks are often initialized to ones
   (or small constants) so every neuron starts with some activation.


#### `np.eye` — Create an identity matrix
**ML use cases:** identity matrix in linear algebra, one-hot encoding labels, regularization.

In [159]:
# --- Example 1: 3×3 identity matrix ---
I3 = np.eye(3)   # 'eye' sounds like 'I' for Identity!
print("3×3 Identity matrix:")
print(I3)

# --- Example 2: 5×5 identity (for one-hot encoding 5 classes) ---
one_hot_template = np.eye(5)
print("\n5×5 Identity (one-hot for 5 classes):")
print(one_hot_template)
print("\nRow 0 = class 0:", one_hot_template[0])  # [1, 0, 0, 0, 0]
print("Row 3 = class 3:", one_hot_template[3])     # [0, 0, 0, 1, 0]

# --- Example 3: Non-square 'identity' ---
rect_eye = np.eye(3, 5)   # 3 rows, 5 columns — 1s on the diagonal
print("\nNon-square eye (3×5):")
print(rect_eye)

print("\n💡 ML context: If you have labels [0, 2, 1, 0] for 3 classes,")
print("   np.eye(3)[[0, 2, 1, 0]] gives you the one-hot encoded matrix!")

3×3 Identity matrix:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

5×5 Identity (one-hot for 5 classes):
[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]

Row 0 = class 0: [1. 0. 0. 0. 0.]
Row 3 = class 3: [0. 0. 0. 1. 0.]

Non-square eye (3×5):
[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]]

💡 ML context: If you have labels [0, 2, 1, 0] for 3 classes,
   np.eye(3)[[0, 2, 1, 0]] gives you the one-hot encoded matrix!


In [160]:
# Quick demo of the one-hot trick mentioned above
labels = [0, 2, 1, 0]                   # class labels for 4 samples
one_hot = np.eye(3)[labels]              # fancy indexing into identity matrix
print("Labels:", labels)
print("One-hot encoded:")
print(one_hot)

Labels: [0, 2, 1, 0]
One-hot encoded:
[[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]


#### `np.full` — Create an array filled with ANY constant
**ML use cases:** initializing arrays with specific values, creating constant masks.

In [164]:
# --- Example 1: Fill with a specific number ---
f1 = np.full(5, 3.14)         # 5 elements, all π
print("Filled with π:", f1)

# --- Example 2: 2-D array filled with -1 (common sentinel value) ---
f2 = np.full((3, 3), -1)      # 3×3 matrix of -1s
print("\n3×3 of -1s:")
print(f2)

# --- Example 3: Fill with a large constant (padding token ID) ---
padding = np.full((2, 10), 0, dtype=np.int32)   # 2 sequences, padded to length 10
print("\nPadding array (2 sequences × length 10):")
print(padding)

print("\n💡 ML context: In NLP, sequences shorter than the max length")
print("   are padded with a constant (often 0) to make uniform-length batches.")

Filled with π: [3.14 3.14 3.14 3.14 3.14]

3×3 of -1s:
[[-1 -1 -1]
 [-1 -1 -1]
 [-1 -1 -1]]

Padding array (2 sequences × length 10):
[[0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]]

💡 ML context: In NLP, sequences shorter than the max length
   are padded with a constant (often 0) to make uniform-length batches.


### 3.3 — `np.arange` vs `np.linspace`

Both create sequences of numbers, but they work differently:

| Function | You specify… | You get… |
|---|---|---|
| `np.arange(start, stop, step)` | The **step size** | However many values fit |
| `np.linspace(start, stop, num)` | The **number of values** | Evenly spaced between start and stop |

**Rule of thumb:**
- Use `arange` when you care about the **step size** (e.g., "every 0.5 units")
- Use `linspace` when you care about the **number of points** (e.g., "give me exactly 50 points between 0 and 1")

In [165]:
# ========== np.arange examples ==========

# --- Example 1: Simple range (like Python's range, but returns an array) ---
a1 = np.arange(10)            # 0 to 9
print("arange(10)       :", a1)

# --- Example 2: Start, stop, step ---
a2 = np.arange(0, 1, 0.2)    # 0.0, 0.2, 0.4, 0.6, 0.8 (stop is EXCLUDED)
print("arange(0, 1, 0.2):", a2)

# --- Example 3: Negative step (counting down) ---
a3 = np.arange(10, 0, -2)    # 10, 8, 6, 4, 2 (stop is EXCLUDED)
print("arange(10, 0, -2):", a3)

print("\n⚠️  Note: np.arange EXCLUDES the stop value (just like Python's range).")

arange(10)       : [0 1 2 3 4 5 6 7 8 9]
arange(0, 1, 0.2): [0.  0.2 0.4 0.6 0.8]
arange(10, 0, -2): [10  8  6  4  2]

⚠️  Note: np.arange EXCLUDES the stop value (just like Python's range).


In [ ]:
# ========== np.linspace examples ==========

# --- Example 1: 5 evenly spaced points between 0 and 1 ---
l1 = np.linspace(0, 1, 5)    # start=0, stop=1, num=5 (stop IS INCLUDED)
print("linspace(0, 1, 5)  :", l1)

# --- Example 2: 10 points for a smooth curve between -π and π ---
l2 = np.linspace(-np.pi, np.pi, 10)  # great for plotting sine/cosine
print("linspace(-π, π, 10):", np.round(l2, 3))

# --- Example 3: 100 points for learning rate schedule ---
lr_schedule = np.linspace(0.01, 0.001, 100)   # decay from 0.01 to 0.001
print("Learning rate schedule (first 5):", lr_schedule[:5])
print("Learning rate schedule (last 5) :", lr_schedule[-5:])

print("\n⚠️  Note: np.linspace INCLUDES both start AND stop by default.")
print("💡 ML context: linspace is great for creating learning rate schedules,")
print("   threshold ranges for evaluation, or x-values for plotting.")

### 3.4 — Random Arrays

Random numbers are **everywhere** in ML:
- **Weight initialization** — neural network weights start as small random numbers
- **Data shuffling** — randomizing the order of training samples each epoch
- **Dropout** — randomly "turning off" neurons during training
- **Train/test splits** — randomly dividing data into subsets

NumPy's `np.random` module is your toolkit for all of this.

#### `np.random.seed` — Reproducibility

Before we generate any random numbers, let's talk about **seeds**.

In [ ]:
# --- Why seeds matter ---
# Without a seed, you get DIFFERENT numbers every time:
print("Without seed (run this cell multiple times to see different results):")
print("  Random numbers:", np.random.rand(3))

# With a seed, you get the SAME numbers every time:
np.random.seed(42)   # 42 is the most popular seed (thanks to Hitchhiker's Guide!)
print("\nWith seed=42 (always the same):")
print("  Random numbers:", np.random.rand(3))

np.random.seed(42)   # reset the seed
print("  Random numbers:", np.random.rand(3))  # exact same output!

print("\n💡 ML context: ALWAYS set a seed when doing experiments.")
print("   If your results aren't reproducible, you can't debug them,")
print("   and reviewers/colleagues can't verify them.")

#### `np.random.rand` — Uniform Distribution [0, 1)

Every value between 0 and 1 is **equally likely**. Think of it like spinning a fair roulette wheel on the interval [0, 1).

In [ ]:
np.random.seed(42)  # for reproducibility

# --- Example 1: 5 random numbers between 0 and 1 ---
r1 = np.random.rand(5)
print("5 uniform random numbers:", r1)

# --- Example 2: 3×4 matrix of random numbers ---
r2 = np.random.rand(3, 4)  # note: dimensions as separate args, NOT a tuple
print("\n3×4 random matrix:")
print(np.round(r2, 3))     # round for readability

# --- Example 3: Scale to any range [a, b] using formula: a + (b - a) * rand ---
a, b = 10, 20               # we want random numbers between 10 and 20
r3 = a + (b - a) * np.random.rand(5)
print(f"\n5 random numbers between {a} and {b}:", np.round(r3, 2))

print("\n💡 ML context: Uniform random numbers are used for random")
print("   initialization, dropout masks, and random data augmentation.")

#### `np.random.randn` — Standard Normal (Gaussian) Distribution

Values follow a **bell curve** centered at 0 with standard deviation 1. Most values cluster near 0; extreme values are rare.

This is the distribution used for **weight initialization** in most neural networks (e.g., Xavier/He initialization).

In [ ]:
np.random.seed(42)

# --- Example 1: 5 numbers from standard normal distribution ---
n1 = np.random.randn(5)
print("5 standard normal numbers:", np.round(n1, 3))
print("(Most values are between -2 and +2)")

# --- Example 2: 3×3 matrix (like a small weight matrix) ---
n2 = np.random.randn(3, 3)
print("\n3×3 normal random matrix:")
print(np.round(n2, 3))

# --- Example 3: Scale to mean=μ, std=σ using formula: μ + σ * randn ---
mu, sigma = 170, 10   # average height 170cm, std dev 10cm
heights = mu + sigma * np.random.randn(1000)  # simulate 1000 people's heights
print(f"\nSimulated heights (mean={mu}, std={sigma}):")
print(f"  Actual mean  : {heights.mean():.1f} cm")
print(f"  Actual std   : {heights.std():.1f} cm")
print(f"  Min height   : {heights.min():.1f} cm")
print(f"  Max height   : {heights.max():.1f} cm")

print("\n💡 ML context: Neural network weights are typically initialized from")
print("   a normal distribution scaled by the number of inputs (He/Xavier init).")

#### `np.random.randint` — Random Integers

Generate random integers in a range. Great for simulating categorical data, creating random indices, or generating labels.

In [ ]:
np.random.seed(42)

# --- Example 1: 10 random integers between 0 and 9 (like digit labels) ---
digits = np.random.randint(0, 10, size=10)   # low=0, high=10 (exclusive), 10 values
print("Random digit labels:", digits)

# --- Example 2: Random binary labels (0 or 1) ---
binary_labels = np.random.randint(0, 2, size=15)  # high=2 means {0, 1}
print("Binary labels:", binary_labels)

# --- Example 3: 4×4 matrix of random integers (like a confusion matrix sketch) ---
rand_matrix = np.random.randint(0, 100, size=(4, 4))
print("\nRandom 4×4 integer matrix:")
print(rand_matrix)

print("\n💡 ML context: randint is handy for generating synthetic class labels,")
print("   random indices for sampling, and creating test/dummy data.")

### ✏️ Exercise 1: Creating Arrays

Try creating the following arrays. Write your code in the cell below, then check the solution in the next cell.

1. Create a 1-D array containing the first 8 **even numbers** starting from 2 (i.e., [2, 4, 6, ..., 16])
2. Create a 4×4 matrix filled with the number 7
3. Create an array of 6 equally spaced values between 0 and 5 (inclusive)
4. Create a 3×3 matrix of random integers between 1 and 10 (inclusive) — remember to set seed=99
5. Create a 1-D array of 5 numbers drawn from a normal distribution with mean=50 and std=10 — remember to set seed=99

In [ ]:
# ============================================
# ✏️ YOUR CODE HERE — Exercise 1
# ============================================

# 1. First 8 even numbers starting from 2


# 2. 4x4 matrix of 7s


# 3. 6 equally spaced values between 0 and 5


# 4. 3x3 random integers between 1 and 10 (seed=99)


# 5. Normal distribution: mean=50, std=10, 5 values (seed=99)



In [ ]:
# ============================================
# ✅ SOLUTION — Exercise 1
# ============================================

# 1. First 8 even numbers starting from 2
even_numbers = np.arange(2, 18, 2)   # start=2, stop=18 (excluded), step=2
print("1. Even numbers:", even_numbers)

# 2. 4×4 matrix of 7s
sevens = np.full((4, 4), 7)          # fill a 4×4 matrix with 7
print("\n2. Matrix of 7s:")
print(sevens)

# 3. 6 equally spaced values between 0 and 5
spaced = np.linspace(0, 5, 6)        # 6 values from 0 to 5 inclusive
print("\n3. Linspace (0 to 5, 6 values):", spaced)

# 4. 3×3 random integers between 1 and 10 (inclusive)
np.random.seed(99)                    # set seed for reproducibility
rand_ints = np.random.randint(1, 11, size=(3, 3))  # high=11 because upper bound is exclusive
print("\n4. Random integers (1–10):")
print(rand_ints)

# 5. Normal distribution: mean=50, std=10
np.random.seed(99)
normal_vals = 50 + 10 * np.random.randn(5)  # mean + std * standard_normal
print("\n5. Normal (mean=50, std=10):", np.round(normal_vals, 2))

---
## Section 4: Array Attributes

Every NumPy array carries metadata about itself. The four attributes you will use **constantly** are:

| Attribute | What it tells you | Example |
|---|---|---|
| `.shape` | Dimensions as a tuple | `(100, 5)` → 100 rows, 5 columns |
| `.ndim` | Number of dimensions | `2` → it's a matrix |
| `.size` | Total number of elements | `500` → 100 × 5 |
| `.dtype` | Data type of each element | `float64` |

### Why shape matters

> **"In ML, shape errors are the #1 source of bugs."** — Every ML practitioner, ever

When your model expects input of shape `(batch_size, 784)` and you accidentally pass `(784,)`, you get a cryptic error. Learning to read and reason about shapes will save you **hours** of debugging.

In [ ]:
# Let's create three arrays of different dimensionality
vec = np.array([10, 20, 30, 40, 50])                # 1-D: a vector
mat = np.array([[1, 2, 3], [4, 5, 6]])              # 2-D: a matrix
cube = np.random.randint(0, 10, size=(2, 3, 4))     # 3-D: like a small image batch

# --- .shape: the most important attribute ---
print("=== .shape ===")
print(f"Vector shape : {vec.shape}")    # (5,)       — one dimension, 5 elements
print(f"Matrix shape : {mat.shape}")    # (2, 3)     — 2 rows, 3 columns
print(f"3-D shape    : {cube.shape}")   # (2, 3, 4)  — 2 'layers', 3 rows, 4 columns

# --- .ndim: number of dimensions ---
print("\n=== .ndim ===")
print(f"Vector ndim  : {vec.ndim}")     # 1
print(f"Matrix ndim  : {mat.ndim}")     # 2
print(f"3-D ndim     : {cube.ndim}")    # 3

# --- .size: total element count ---
print("\n=== .size ===")
print(f"Vector size  : {vec.size}")     # 5
print(f"Matrix size  : {mat.size}")     # 6  (2 × 3)
print(f"3-D size     : {cube.size}")    # 24 (2 × 3 × 4)

# --- .dtype: element data type ---
print("\n=== .dtype ===")
print(f"Vector dtype : {vec.dtype}")    # int64 (default for integers)
print(f"Matrix dtype : {mat.dtype}")    # int64
print(f"3-D dtype    : {cube.dtype}")   # int64 (randint produces ints)

### Reading Shapes in ML Context

Shapes have **meaning**. Here is how to interpret common shapes you will encounter:

| Shape | What it typically represents |
|---|---|
| `(n,)` | A vector of `n` values (e.g., labels for n samples) |
| `(n, d)` | A dataset: `n` samples, `d` features each |
| `(n, h, w)` | A batch of `n` grayscale images, each `h` × `w` pixels |
| `(n, h, w, c)` | A batch of `n` color images, `h` × `w` pixels, `c` channels (usually 3 for RGB) |
| `(n, c, h, w)` | Same but "channels first" (PyTorch convention) |

Let's see these in practice:

In [ ]:
# --- Example 1: A tabular dataset (samples × features) ---
np.random.seed(42)
dataset = np.random.randn(100, 5)   # 100 samples, 5 features each
print(f"Dataset shape: {dataset.shape}")
print(f"  → {dataset.shape[0]} samples, {dataset.shape[1]} features")

# --- Example 2: A batch of grayscale images (batch × height × width) ---
gray_images = np.random.rand(32, 28, 28)  # 32 images, each 28×28 (like MNIST)
print(f"\nGrayscale image batch shape: {gray_images.shape}")
print(f"  → {gray_images.shape[0]} images, each {gray_images.shape[1]}×{gray_images.shape[2]} pixels")

# --- Example 3: A batch of color images (batch × height × width × channels) ---
color_images = np.random.rand(16, 224, 224, 3)  # 16 images, 224×224, RGB
print(f"\nColor image batch shape: {color_images.shape}")
print(f"  → {color_images.shape[0]} images")
print(f"  → each {color_images.shape[1]}×{color_images.shape[2]} pixels")
print(f"  → {color_images.shape[3]} color channels (RGB)")
print(f"  → Total numbers: {color_images.size:,}")  # ~2.4 million numbers!

### Data Types (`dtype`) — Why They Matter

NumPy arrays are **homogeneous**: every element has the same type. The most common types:

| dtype | Bytes | Range / Precision | When to use |
|---|---|---|---|
| `int32` | 4 | ±2 billion | Integer data, labels, indices |
| `int64` | 8 | ±9.2 quintillion | Default for integers |
| `float32` | 4 | ~7 decimal digits | **ML standard** — GPU-friendly, memory-efficient |
| `float64` | 8 | ~15 decimal digits | Default for floats — high precision |
| `bool` | 1 | True/False | Masks, conditions |

### Why `float32` is the ML standard

> **`float32` uses half the memory of `float64` and is what GPUs are optimized for.** Most ML frameworks (TensorFlow, PyTorch) default to `float32`. The extra precision of `float64` is almost never needed in ML — your model's noise dwarfs the rounding error.

In [ ]:
# --- Memory comparison ---
arr_f64 = np.random.randn(1000, 1000)                  # default: float64
arr_f32 = np.random.randn(1000, 1000).astype(np.float32)  # convert to float32

print(f"float64 array: {arr_f64.nbytes / 1e6:.1f} MB")   # 8 bytes × 1M = 8 MB
print(f"float32 array: {arr_f32.nbytes / 1e6:.1f} MB")   # 4 bytes × 1M = 4 MB
print(f"\nfloat32 uses {arr_f32.nbytes / arr_f64.nbytes * 100:.0f}% of the memory — HALF!")

# --- dtype matters for correctness ---
int_arr = np.array([1, 2, 3, 4, 5], dtype=np.int32)
float_arr = np.array([1, 2, 3, 4, 5], dtype=np.float32)
print(f"\nInteger array: {int_arr}, dtype={int_arr.dtype}")
print(f"Float array  : {float_arr}, dtype={float_arr.dtype}")

### Changing dtypes with `.astype()` — Watch Out for Truncation!

You can convert between types using `.astype()`, but be careful: converting float to int **truncates** (drops the decimal part without rounding).

In [ ]:
# --- Safe conversion: int → float (no data loss) ---
int_data = np.array([10, 20, 30])
float_data = int_data.astype(np.float32)      # safe: int → float
print("int → float (safe):", float_data, "dtype:", float_data.dtype)

# --- Dangerous conversion: float → int (TRUNCATION!) ---
float_data2 = np.array([1.9, 2.7, 3.1, -0.8])
int_data2 = float_data2.astype(np.int32)       # TRUNCATES, does NOT round!
print("\nfloat → int (TRUNCATION):", int_data2)
print("Original was:             ", float_data2)
print("⚠️  Notice: 1.9 became 1, 2.7 became 2, -0.8 became 0")
print("   This is truncation toward zero, NOT rounding!")

# --- How to round properly before converting ---
rounded_then_int = np.round(float_data2).astype(np.int32)  # round FIRST, then convert
print("\nRound then convert:", rounded_then_int)  # [2, 3, 3, -1]

# --- float64 → float32 (tiny precision loss, usually fine for ML) ---
precise = np.array([1.123456789012345])
less_precise = precise.astype(np.float32)
print(f"\nfloat64: {precise[0]:.15f}")
print(f"float32: {float(less_precise[0]):.15f}")
print("(Only ~7 digits of precision in float32 — perfectly fine for ML)")

### ✏️ Exercise 2: Shape Detective

For each description below, **predict the shape** of the resulting array. Write your answer as a tuple (e.g., `(3, 4)`), then check the solution.

1. A dataset with 500 samples, each having 10 features
2. A batch of 64 color images, each 32×32 pixels with 3 color channels (channels last)
3. The output of `np.zeros(8)`
4. The output of `np.eye(6)`
5. A 3-D array created by `np.ones((4, 3, 2))`
6. Labels for 200 samples (just a list of class numbers)

In [ ]:
# ============================================
# ✏️ YOUR CODE HERE — Exercise 2
# ============================================
# Write your shape predictions as comments, then create the arrays to verify:

# 1. 500 samples × 10 features → predicted shape: ???


# 2. 64 color images, 32×32, 3 channels → predicted shape: ???


# 3. np.zeros(8) → predicted shape: ???


# 4. np.eye(6) → predicted shape: ???


# 5. np.ones((4, 3, 2)) → predicted shape: ???


# 6. Labels for 200 samples → predicted shape: ???



In [ ]:
# ============================================
# ✅ SOLUTION — Exercise 2
# ============================================

# 1. Dataset: 500 samples × 10 features
data = np.random.randn(500, 10)
print(f"1. Dataset shape: {data.shape}")            # (500, 10)

# 2. Color images: 64 batch × 32 height × 32 width × 3 channels
images = np.random.rand(64, 32, 32, 3)
print(f"2. Image batch shape: {images.shape}")       # (64, 32, 32, 3)

# 3. np.zeros(8)
z = np.zeros(8)
print(f"3. zeros(8) shape: {z.shape}")               # (8,) — 1-D!

# 4. np.eye(6)
e = np.eye(6)
print(f"4. eye(6) shape: {e.shape}")                 # (6, 6) — always square

# 5. np.ones((4, 3, 2))
o = np.ones((4, 3, 2))
print(f"5. ones((4,3,2)) shape: {o.shape}")          # (4, 3, 2)

# 6. Labels for 200 samples — just a 1-D vector
labels = np.random.randint(0, 5, size=200)
print(f"6. Labels shape: {labels.shape}")            # (200,)

---
## Section 5: Creating Arrays for ML

Now let's put everything together and create the kind of arrays you will see in real ML code. This section bridges the gap between "NumPy tutorial" and "actually doing ML."

### 5.1 — Creating a Mock Dataset

Imagine you are working with a housing dataset: 100 houses, each described by 5 features (square footage, bedrooms, bathrooms, age, distance to city center).

In [ ]:
np.random.seed(42)

n_samples = 100    # number of houses
n_features = 5     # number of features per house

# Create feature matrix X — each row is one house, each column is one feature
X = np.random.randn(n_samples, n_features)  # standardized features (mean≈0, std≈1)

# Create target vector y — the house price we want to predict
y = np.random.rand(n_samples) * 500_000 + 100_000  # prices between $100K and $600K

print(f"Feature matrix X:")
print(f"  Shape: {X.shape}")                  # (100, 5)
print(f"  dtype: {X.dtype}")                  # float64
print(f"  First 3 rows:\n{np.round(X[:3], 3)}")

print(f"\nTarget vector y:")
print(f"  Shape: {y.shape}")                   # (100,)
print(f"  dtype: {y.dtype}")                   # float64
print(f"  First 5 prices: ${y[:5].astype(int)}")

print(f"\n💡 This X, y pair is exactly what you pass to scikit-learn:")
print(f"   model.fit(X, y)  # X is (n_samples, n_features), y is (n_samples,)")

### 5.2 — Creating Weight Matrices for a Simple Neural Network

A neural network is essentially a series of **matrix multiplications**. Each layer has a weight matrix `W` and a bias vector `b`. Let's create them for a simple 3-layer network:

- Input: 784 features (like a flattened 28×28 MNIST image)
- Hidden layer 1: 128 neurons
- Hidden layer 2: 64 neurons
- Output: 10 classes (digits 0–9)

In [ ]:
np.random.seed(42)

# Layer dimensions
input_dim = 784     # 28 × 28 pixels flattened
hidden1 = 128       # first hidden layer size
hidden2 = 64        # second hidden layer size
output_dim = 10     # 10 digit classes

# --- Weight matrices (random initialization, scaled by layer size) ---
# He initialization: scale by sqrt(2 / fan_in) — a common best practice
W1 = np.random.randn(input_dim, hidden1) * np.sqrt(2.0 / input_dim)   # (784, 128)
W2 = np.random.randn(hidden1, hidden2) * np.sqrt(2.0 / hidden1)       # (128, 64)
W3 = np.random.randn(hidden2, output_dim) * np.sqrt(2.0 / hidden2)    # (64, 10)

# --- Bias vectors (initialized to zeros) ---
b1 = np.zeros(hidden1)     # (128,)
b2 = np.zeros(hidden2)     # (64,)
b3 = np.zeros(output_dim)  # (10,)

print("Neural Network Architecture:")
print(f"  Input        : {input_dim} features")
print(f"  W1 shape     : {W1.shape}  (input → hidden1)")
print(f"  b1 shape     : {b1.shape}")
print(f"  W2 shape     : {W2.shape}   (hidden1 → hidden2)")
print(f"  b2 shape     : {b2.shape}")
print(f"  W3 shape     : {W3.shape}    (hidden2 → output)")
print(f"  b3 shape     : {b3.shape}")

total_params = W1.size + b1.size + W2.size + b2.size + W3.size + b3.size
print(f"\n  Total parameters: {total_params:,}")  # ~109K parameters!

print("\n💡 Every deep learning framework creates arrays like these under the hood.")
print("   Understanding their shapes is essential for building and debugging models.")

### 5.3 — Creating One-Hot Encoded Labels

Classification problems need labels. Raw labels might be `[0, 3, 1, 2]`, but many loss functions expect **one-hot encoding** where each label becomes a vector of 0s with a single 1.

In [ ]:
np.random.seed(42)

n_samples = 8      # 8 samples for demonstration
n_classes = 4      # 4 possible classes (e.g., cat, dog, bird, fish)

# --- Create raw integer labels ---
raw_labels = np.random.randint(0, n_classes, size=n_samples)
print("Raw labels:", raw_labels)

# --- Method 1: One-hot encoding using np.eye (the elegant way) ---
one_hot = np.eye(n_classes)[raw_labels]   # index into identity matrix
print("\nOne-hot encoded labels:")
print(one_hot)

# --- Method 2: Manual one-hot encoding (for understanding) ---
one_hot_manual = np.zeros((n_samples, n_classes))   # start with all zeros
for i, label in enumerate(raw_labels):               # fill in the 1s
    one_hot_manual[i, label] = 1.0                   # set the correct position to 1

print("\nManual one-hot (same result):")
print(one_hot_manual)

# Verify they're identical
print(f"\nBoth methods give same result: {np.array_equal(one_hot, one_hot_manual)}")
print(f"One-hot shape: {one_hot.shape}")  # (8, 4) — 8 samples, 4 classes

### ✏️ Exercise 3: Build an ML Scenario

Imagine you are preparing data for a simple image classification task:

1. **Create a batch of fake grayscale images**: 16 images, each 8×8 pixels, with random pixel values between 0 and 255 (integers). Set seed=123.
2. **Create the corresponding labels**: 16 random integer labels for 5 classes (0–4). Set seed=123.
3. **One-hot encode the labels** using the `np.eye` trick.
4. **Normalize the images** to float32 values between 0 and 1 (divide by 255 and cast to float32).
5. **Print** the shape and dtype of every array you created.

In [ ]:
# ============================================
# ✏️ YOUR CODE HERE — Exercise 3
# ============================================

# 1. Batch of 16 grayscale images (8×8), pixel values 0-255, seed=123


# 2. Labels for 5 classes, seed=123


# 3. One-hot encode the labels


# 4. Normalize images to float32 [0, 1]


# 5. Print shapes and dtypes



In [ ]:
# ============================================
# ✅ SOLUTION — Exercise 3
# ============================================

# 1. Batch of 16 grayscale images, 8×8 pixels, values 0-255
np.random.seed(123)
images = np.random.randint(0, 256, size=(16, 8, 8))  # 256 is exclusive → 0 to 255
print(f"1. Images shape: {images.shape}, dtype: {images.dtype}")

# 2. Labels for 5 classes
np.random.seed(123)
labels = np.random.randint(0, 5, size=16)             # 16 labels, classes 0-4
print(f"2. Labels shape: {labels.shape}, dtype: {labels.dtype}")
print(f"   Labels: {labels}")

# 3. One-hot encode
one_hot_labels = np.eye(5)[labels]                     # 5 classes → eye(5)
print(f"3. One-hot shape: {one_hot_labels.shape}, dtype: {one_hot_labels.dtype}")
print(f"   First 3 one-hot vectors:\n{one_hot_labels[:3]}")

# 4. Normalize to float32 [0, 1]
images_normalized = (images / 255.0).astype(np.float32)  # divide by 255, cast to float32
print(f"4. Normalized images shape: {images_normalized.shape}, dtype: {images_normalized.dtype}")
print(f"   Pixel range: [{images_normalized.min():.4f}, {images_normalized.max():.4f}]")

# 5. Summary
print(f"\n--- Summary ---")
print(f"{'Array':<20} {'Shape':<20} {'dtype':<10} {'Size (bytes)':<12}")
print("-" * 62)
for name, arr in [("images", images), ("labels", labels),
                   ("one_hot_labels", one_hot_labels), ("images_norm", images_normalized)]:
    print(f"{name:<20} {str(arr.shape):<20} {str(arr.dtype):<10} {arr.nbytes:<12}")

---
## Summary — Key Takeaways

Congratulations! You now have a solid foundation in NumPy array creation and attributes. Here's what we covered and **why it matters for ML**:

### 🔑 Key Concepts

| Concept | Why It Matters for ML |
|---|---|
| **Vectorization** | Makes ML computationally feasible — 50-100× faster than Python loops |
| **Array creation** | Every ML pipeline starts by loading/creating data as NumPy arrays |
| **`np.random.seed`** | Reproducibility is non-negotiable in ML research and production |
| **Shape `(n, d)`** | Understanding shapes prevents the #1 source of ML bugs |
| **dtype `float32`** | The standard for ML — saves memory, GPU-compatible |
| **One-hot encoding** | Required by many loss functions (cross-entropy, etc.) |
| **Weight initialization** | Proper random init (He/Xavier) is critical for training neural networks |

### 📐 Shape Cheat Sheet

```
Labels:      (n_samples,)
Dataset:     (n_samples, n_features)
Grayscale:   (batch, height, width)
Color (TF):  (batch, height, width, channels)
Color (PT):  (batch, channels, height, width)
```

### ⚡ Quick Reference

```python
np.array([...])          # from a list
np.zeros((r, c))         # all zeros
np.ones((r, c))          # all ones
np.eye(n)                # identity matrix
np.full((r, c), val)     # fill with constant
np.arange(start, stop, step)   # by step size
np.linspace(start, stop, num)  # by count
np.random.randn(r, c)   # normal distribution
np.random.rand(r, c)    # uniform [0, 1)
np.random.randint(lo, hi, size) # random integers
```

### What's Next?

In the next notebook, we'll learn about **array operations** — indexing, slicing, reshaping, and the mathematical operations that make NumPy the powerhouse it is. These are the tools you'll use to *manipulate* the arrays you just learned to create. See you there!